# Aşama 12-v2 — Hızlı Final Test
Progress bar + model/seed bazında resume. ConvNeXt/ResNet batch 64, RETFound batch 8. Kernel: **retfound**. Tamamlanan model/seed `DONE.json` sayesinde tekrar çalışmaz.


In [1]:
from pathlib import Path
import time,json,importlib.util,numpy as np,torch
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models
from sklearn.metrics import f1_score,balanced_accuracy_score,accuracy_score
from tqdm.auto import tqdm
R=Path.cwd();D=R/"wbc_project/data/processed/phase2_test";O=R/"outputs/final_test_v2";O.mkdir(parents=True,exist_ok=True)
dev=torch.device("cuda");tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]);ds=datasets.ImageFolder(D,tf)
def mk(n):
 if n=="convnext_tiny":
  m=models.convnext_tiny(weights=None);m.classifier[2]=torch.nn.Linear(m.classifier[2].in_features,13);return m
 if n=="resnet50":
  m=models.resnet50(weights=None);m.fc=torch.nn.Linear(m.fc.in_features,13);return m
 q=importlib.util.spec_from_file_location("rv",R/"RETFound/models_vit.py");x=importlib.util.module_from_spec(q);q.loader.exec_module(x);return x.RETFound_mae(num_classes=13,drop_path_rate=0.,global_pool=True)
def path(n,s):return R/(f"outputs/final/{n}/seed{s}/{n}_seed{s}_best.pt" if n!="retfound_fullft" else f"outputs/final/retfound_fullft/seed{s}/retfound_fullft_seed{s}_best.pt")
@torch.inference_mode()
def go(n,s):
 d=O/n/f"seed{s}";d.mkdir(parents=True,exist_ok=True);done=d/"DONE.json"
 if done.exists():print("✓ atlandı",n,s);return json.loads(done.read_text())
 bs=64 if n!="retfound_fullft" else 8;L=DataLoader(ds,bs,shuffle=False,num_workers=0,pin_memory=True);m=mk(n);c=torch.load(path(n,s),map_location="cpu",weights_only=False);m.load_state_dict(c["model_state_dict"]);m.to(dev).eval()
 Y=[];P=[];Q=[];t=time.time()
 for a,b in tqdm(L,desc=f"{n} seed{s}"):
  a=a.to(dev)
  with torch.autocast("cuda",dtype=torch.float16):z=m(a)
  q=torch.softmax(z.float(),1).cpu().numpy();Q.append(q);P+=q.argmax(1).tolist();Y+=b.tolist()
 Y=np.array(Y);P=np.array(P);Q=np.concatenate(Q);np.save(d/"probabilities.npy",Q);np.save(d/"true.npy",Y);np.save(d/"pred.npy",P)
 r={"model":n,"seed":s,"macro_f1":float(f1_score(Y,P,average="macro",zero_division=0)),"balanced_accuracy":float(balanced_accuracy_score(Y,P)),"accuracy":float(accuracy_score(Y,P)),"minutes":(time.time()-t)/60};done.write_text(json.dumps(r,indent=2));print("✓",r);del m;torch.cuda.empty_cache();return r
rows=[go(n,s) for n in ["convnext_tiny","resnet50","retfound_fullft"] for s in [42,1,7]]
print("\n=== FINAL TEST mean ± sample SD ===")
for n in ["convnext_tiny","resnet50","retfound_fullft"]:
 for k in ["macro_f1","balanced_accuracy","accuracy"]:
  a=np.array([r[k] for r in rows if r["model"]==n]);print(n,k,f"{a.mean():.4f} ± {a.std(ddof=1):.4f}")


C:\Users\sarah\.conda\envs\retfound\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
convnext_tiny seed42: 100%|██████████| 515/515 [05:57<00:00,  1.44it/s]


✓ {'model': 'convnext_tiny', 'seed': 42, 'macro_f1': 0.6463867297197994, 'balanced_accuracy': 0.6522602365519521, 'accuracy': 0.9458614198093156, 'minutes': 5.951138397057851}


convnext_tiny seed1: 100%|██████████| 515/515 [05:21<00:00,  1.60it/s]


✓ {'model': 'convnext_tiny', 'seed': 1, 'macro_f1': 0.6329458935288568, 'balanced_accuracy': 0.6123162354120354, 'accuracy': 0.9542114532094492, 'minutes': 5.366373976071675}


convnext_tiny seed7: 100%|██████████| 515/515 [05:25<00:00,  1.58it/s]


✓ {'model': 'convnext_tiny', 'seed': 7, 'macro_f1': 0.6306776819894006, 'balanced_accuracy': 0.6114763464400311, 'accuracy': 0.9480779741300783, 'minutes': 5.432854123910269}


... (8 outputs left out)

retfound_fullft seed1: 100%|██████████| 4117/4117 [09:43<00:00,  7.06it/s]


✓ {'model': 'retfound_fullft', 'seed': 1, 'macro_f1': 0.6051410723475166, 'balanced_accuracy': 0.5997947735621729, 'accuracy': 0.9398493957612194, 'minutes': 9.725625201066334}


retfound_fullft seed7: 100%|██████████| 4117/4117 [09:52<00:00,  6.95it/s]

✓ {'model': 'retfound_fullft', 'seed': 7, 'macro_f1': 0.6115952398294393, 'balanced_accuracy': 0.596131404433009, 'accuracy': 0.9406388534645048, 'minutes': 9.871137710412343}

=== FINAL TEST mean ± sample SD ===
convnext_tiny macro_f1 0.6367 ± 0.0085
convnext_tiny balanced_accuracy 0.6254 ± 0.0233
convnext_tiny accuracy 0.9494 ± 0.0043
resnet50 macro_f1 0.6466 ± 0.0143
resnet50 balanced_accuracy 0.6353 ± 0.0181
resnet50 accuracy 0.9486 ± 0.0041
retfound_fullft macro_f1 0.6107 ± 0.0052
retfound_fullft balanced_accuracy 0.5977 ± 0.0019
retfound_fullft accuracy 0.9407 ± 0.0009
